In [ ]:
import pandas as pd
from scipy import stats
import numpy as np

import pandas as pd
from scipy import stats
import numpy as np
from sklearn.metrics import f1_score, mean_absolute_error, mean_squared_error
from math import sqrt

def score2label(score, res=0, size=7, m=0.49, M=1.69):
    step = (M-m) / (size)
    if score < step + m:
        return 1+res
    else:
        return score2label(score-step, res+1, size, m, M)

def sigmoid(z):
    return 1/(1 + np.exp(-z))

def quadratic_weighted_kappa(y_true, y_pred, s_min, s_max):
    """
    Calculate the Quadratic Weighted Kappa (QWK) between two raters.

    Args:
    y_true (np.array): The true labels or ratings (Rater B).
    y_pred (np.array): The predicted labels or ratings (Rater A).
    L (int): The number of possible score categories.

    Returns:
    float: The QWK score.
    """

    # Create the weights matrix
    w = np.zeros((s_max-s_min+1, s_max-s_min+1))
    for i in range(s_max-s_min+1):
        for j in range(s_max-s_min+1):
            w[i, j] = ((i - j) ** 2) / ((s_max - s_min) ** 2)

    # Calculate the O (observed) matrix
    O = np.histogram2d(y_true, y_pred, bins=s_max-s_min+1, range=[[s_min, s_max], [s_min, s_max]])[0]

    # Calculate the E (expected) matrix
    hist_true = np.histogram(y_true, bins=s_max-s_min+1, range=[s_min, s_max])[0]
    hist_pred = np.histogram(y_pred, bins=s_max-s_min+1, range=[s_min, s_max])[0]
    E = np.outer(hist_true, hist_pred) / len(y_true)

    # Compute QWK
    num = np.sum(w * O)
    denom = np.sum(w * E)
    kappa = 1 - (num / denom)

    return kappa

S_MAX = 7
S_MIN = 1


df = pd.read_csv(f"data/test.csv", index_col=0)

results = []
for model_path in [
    "data/human",
    "dummy/pred/random",
    "dummy/pred/length",
    "similarity/pred/jaccard",        
    "similarity/pred/cosine",
    "ulra/pred/lf",
    "ulra/pred/ef",
    "ulra/pred/ef_lf",
    "ulra_paper/none/pred/lf",
    "ulra_paper/none/pred/ef",
    "signal_clustering/none/pred",
    "llm/vanilla/pred",
    "llm/cot/pred",  

    "nllf_method/lr/pred/z_score/wo_sf/ef",
    "nllf_method/lr/pred/z_score/wo_sf/nllf",
    "nllf_method/lr/pred/z_score/wo_sf/ef_nllf",
    "nllf_method/lr/pred/llm/wo_sf/ef",
    "nllf_method/lr/pred/llm/wo_sf/nllf",
    "nllf_method/lr/pred/llm/wo_sf/ef_nllf",
    "nllf_method/lr/pred/z_score/w_sf/ef",
    "nllf_method/lr/pred/z_score/w_sf/nllf",
    "nllf_method/lr/pred/z_score/w_sf/ef_nllf",
    "nllf_method/lr/pred/llm/w_sf/ef",
    "nllf_method/lr/pred/llm/w_sf/nllf",
    "nllf_method/lr/pred/llm/w_sf/ef_nllf",


    "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/ef",
    "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/nllf",
    "nllf_method/lr_Z_C/pred/signal_clustering/wo_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/llm/wo_sf/ef",
    "nllf_method/lr_Z_C/pred/llm/wo_sf/nllf",
    "nllf_method/lr_Z_C/pred/llm/wo_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/ulra/wo_sf/ef",
    "nllf_method/lr_Z_C/pred/ulra/wo_sf/nllf",
    "nllf_method/lr_Z_C/pred/ulra/wo_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/ef",
    "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/nllf",
    "nllf_method/lr_Z_C/pred/signal_clustering/w_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/llm/w_sf/ef",
    "nllf_method/lr_Z_C/pred/llm/w_sf/nllf",
    "nllf_method/lr_Z_C/pred/llm/w_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/ulra/w_sf/ef",
    "nllf_method/lr_Z_C/pred/ulra/w_sf/nllf",
    "nllf_method/lr_Z_C/pred/ulra/w_sf/nllf_ef",
    
    "nllf_method/lr_Z_C/pred/signal_clustering/w_r_sf/ef",
    "nllf_method/lr_Z_C/pred/signal_clustering/w_r_sf/nllf",
    "nllf_method/lr_Z_C/pred/signal_clustering/w_r_sf/nllf_ef",

    "nllf_method/lr_Z_C/pred/ulra/w_r_sf/ef",
    "nllf_method/lr_Z_C/pred/ulra/w_r_sf/nllf",
    "nllf_method/lr_Z_C/pred/ulra/w_r_sf/nllf_ef",


    "bert/pred/z_score/wo_sf",
    "bert/pred/llm/wo_sf",
    "bert/pred/z_score/w_sf",
    "bert/pred/llm/w_sf",
]:
    df_test = pd.read_csv(model_path+f"/test.csv", index_col=0)
    df_test["test"] = df["nota"]
    df_test = df_test.dropna()

    y_true = df_test["test"].values
    y_pred = df_test["pred"].values

    m, M = np.min(y_true), np.max(y_true)
    scaled_true = (S_MIN + ((pd.Series(y_true) - m) / (M - m)) * (S_MAX - S_MIN)).values

    m, M = np.min(y_pred), np.max(y_pred)
    if "nllf_method" in model_path:
        m, M = np.percentile(y_pred, 1), np.percentile(y_pred, 99) 
    scale_preds = (S_MIN + ((pd.Series(y_pred) - m) / (M - m)) * (S_MAX - S_MIN)).values if abs(M-m) > 0 else (pd.Series(y_pred) - m)  + (S_MIN + S_MAX) / 2

    if "human" in model_path:
        scale_preds = y_pred
        scaled_true = y_true


    o = {
            "method": model_path.replace("/pred", "").replace("/data", ""), 
            "corr.": np.round(abs(stats.pearsonr(scaled_true, scale_preds)[0]), 4), 
            "mae.": np.round(mean_absolute_error(scaled_true, scale_preds), 4),
            "qwk": np.round(quadratic_weighted_kappa(scaled_true, scale_preds, S_MIN, S_MAX), 4)
        
        }

    results.append(o)
    
    print(o.items())

table = pd.DataFrame(results)
table

In [ ]:
table = table.sort_values("qwk", ascending=False)
table = table.set_index("method")
table

In [ ]:
metric = "qwk"

our_model = "nllf_method/lr_Z_C"

rows = []
o = {
    "method": "Lenght", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["dummy/length"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Jaccard Sim.", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["similarity/jaccard"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Jaccard Sim.", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["similarity/cosine"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "ULRA", 
    "weak_signal": "LF",
    "signal_filtering": "-",
    "text": table.loc["ulra_paper/none/lf"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "ULRA", 
    "weak_signal": "EF",
    "signal_filtering": "-",
    "text": table.loc["ulra/ef"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "ULRA", 
    "weak_signal": "EF+LF",
    "signal_filtering": "-",
    "text": table.loc["ulra/ef_lf"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)


o = {
    "method": "Z-score", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["signal_clustering/none"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "LLM", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["llm/vanilla"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "LLM-CoT", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc["llm/cot"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "Z-score",
    "signal_filtering": "xmark",
    "text": "-",
    "ef": table.loc[f"{our_model}/signal_clustering/wo_sf/ef"][metric],
    "nllf": table.loc[f"{our_model}/signal_clustering/wo_sf/nllf"][metric],
    "ef+nllf": table.loc[f"{our_model}/signal_clustering/wo_sf/nllf_ef"][metric]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "xmark",
    "text": "-",
    "ef": table.loc[f"{our_model}/llm/wo_sf/ef"][metric],
    "nllf": table.loc[f"{our_model}/llm/wo_sf/nllf"][metric],
    "ef+nllf": table.loc[f"{our_model}/llm/wo_sf/nllf_ef"][metric],
}
rows.append(o)


o = {
    "method": "Linear Regression", 
    "weak_signal": "Z-score",
    "signal_filtering": "cmark",
    "text": "-",
    "ef": table.loc[f"{our_model}/signal_clustering/w_r_sf/ef"][metric],
    "nllf": table.loc[f"{our_model}/signal_clustering/w_r_sf/nllf"][metric],
    "ef+nllf": table.loc[f"{our_model}/signal_clustering/w_r_sf/nllf_ef"][metric]
}
rows.append(o)

o = {
    "method": "Linear Regression", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "cmark",
    "text": "-",
    "ef": table.loc[f"{our_model}/llm/w_sf/ef"][metric],
    "nllf": table.loc[f"{our_model}/llm/w_sf/nllf"][metric],
    "ef+nllf": table.loc[f"{our_model}/llm/w_sf/nllf_ef"][metric]
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "Z-score",
    "signal_filtering": "xmark",
    "text": table.loc[f"bert/z_score/wo_sf"][metric],
        "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "xmark",
    "text": table.loc[f"bert/llm/wo_sf"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "Z-score",
    "signal_filtering": "cmark",
    "text": table.loc[f"bert/z_score/w_sf"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

o = {
    "method": "BERT", 
    "weak_signal": "LLM-based signal",
    "signal_filtering": "cmark",
    "text": table.loc[f"bert/llm/w_sf"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)


o = {
    "method": "human", 
    "weak_signal": "None",
    "signal_filtering": "-",
    "text": table.loc[f"data/human"][metric],
    "ef": "-",
    "nllf": "-",
    "ef+nllf": "-"
}
rows.append(o)

pd.DataFrame(rows)